# Redrob AI Recruiter — Full-Stack Sandbox
**Hackathon sandbox** · Candidate ranking + honeypot detection + LLM recruiter UI

| Cell | What it does |
|------|-------------|
| 1 | Set Anthropic API key (needed for Cell 6 web app only) |
| 2 | Install dependencies (`flask`, `anthropic`, `pyngrok`) |
| 3 | Write fixed `rank.py` to disk and load its functions |
| 4 | Upload candidates file → rank offline → print top-10 |
| 5 | Export spec-compliant `submission.csv` |
| 6 | Launch full-stack recruiter web app → get public sandbox URL |

> **Compute compliance (Cells 3-5):** zero GPU, zero network, zero external
> model weights. All 4 critical bugs fixed in this version of `rank.py`:
> UTF-8 encoding, disqualified candidates excluded from pool, exact skill-set
> matching (no substring false-positives), null-safe signal extraction.

In [ ]:
# Cell 1 — Gemini API key
# Only used in Cell 6 (web app search/chat). NOT used during ranking.
import os
try:
    from google.colab import userdata
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY'),
    print(' API key loaded from Colab Secrets')
except Exception:
    os.environ['GEMINI_API_KEY'] = 'AIzaSyAQ.Ab8RN6KYi8-vh2_TnuSeAJzABmkO9VTBWNfM03WOveMUTOQctg'
    print('Using hardcoded key — prefer Colab Secrets (left sidebar )')


In [ ]:
# Cell 2 — Install dependencies
# rank.py needs NOTHING (pure Python stdlib).
# The web app (Cell 6) needs flask, anthropic, pyngrok.
%pip install flask google-genai pydantic pyngrok --quiet
print('flask, google-genai, pydantic, pyngrok installed')


In [ ]:
# Cell 3 — Write fixed rank.py to disk and load its functions
#
# This is the bug-fixed version. Changes from the original:
#   Bug 1: open(candidates, encoding='utf-8')       — no UnicodeDecodeError on Windows
#   Bug 2: continue after honeypot/hard-disq        — excluded candidates never enter scored[]
#   Bug 3: raw_skill_set (set)                      — exact match, no substring false-positives
#   Bug 4: signal or default / is not None          — all 8 signal fields null-safe
#          (includes willing_to_relocate fix added in this revision)

RANK_PY_SOURCE = '''
#!/usr/bin/env python3
"""
Redrob Hackathon — Senior AI Engineer Candidate Ranker
=======================================================
Rule-based multi-signal scorer. No GPU, no network, no external deps.
Runtime: ~19s for 100K candidates on a 4-core CPU.

Trap/honeypot handling (see README for full analysis):
  1. Keyword stuffers     → career description text evidence required, not just skill tags
  2. Plain-language Tier5 → disqualifying titles + no AI career history = hard exclude
  3. Behavioral twins     → behavioral signals used as multiplier, not just additive
  4. ~80 honeypots        → >=10 expert skills = hard disqualify (zero false positives
                             confirmed against legitimate top-100; max legit = 9 expert skills)
                           + expert skills with 0 duration months (>=3 = disqualify)
                           + claimed YoE vs career history divergence (>4yr gap = disqualify)
                           + duration_months vs date span mismatch (>4mo gap = disqualify)
"""

import json, csv, argparse
from datetime import date, datetime

CURRENT_DATE = date(2026, 6, 13)

# ─── JD-derived constants ────────────────────────────────────────────────────

MUST_HAVE_SKILLS = {
    # Embeddings & models
    "sentence-transformers", "openai embeddings", "bge", "e5", "embeddings",
    "dense retrieval", "bi-encoders", "cross-encoders",
    # Vector DBs / hybrid search
    "pinecone", "weaviate", "qdrant", "milvus", "faiss", "opensearch",
    "elasticsearch", "annoy", "scann", "vector database", "vector search",
    "hybrid search", "pgvector", "chromadb",
    # Ranking / IR
    "information retrieval", "ranking", "search", "recommendation systems",
    "learning to rank", "bm25", "ndcg", "mrr", "map",
    "semantic search", "reranking", "re-ranking",
    # LLMs & NLP
    "llm", "large language models", "transformers", "bert", "nlp",
    "natural language processing", "rag", "retrieval augmented generation",
    "fine-tuning", "lora", "qlora", "peft",
    # Core
    "python", "machine learning", "deep learning", "neural networks",
    # Eval
    "a/b testing", "ab testing", "evaluation framework", "offline evaluation",
}

NICE_TO_HAVE_SKILLS = {
    "xgboost", "lightgbm", "gradient boosting", "learning to rank",
    "hugging face", "huggingface", "pytorch", "tensorflow",
    "kubernetes", "docker", "aws", "gcp", "azure", "mlflow", "wandb",
    "spark", "kafka", "redis", "langchain", "llamaindex",
    "distributed systems", "microservices", "open source",
}

CONSULTING_FIRMS = {
    "tcs", "tata consultancy", "infosys", "wipro", "accenture", "cognizant",
    "capgemini", "hcl", "tech mahindra", "mindtree", "mphasis", "hexaware",
    "larsen toubro infotech", "lti", "ltimindtree", "persistent systems",
    "niit technologies", "zensar", "cyient",
}

STRONG_AI_TITLES = {
    "machine learning engineer", "ml engineer", "ai engineer", "nlp engineer",
    "data scientist", "research engineer", "applied scientist",
    "senior ai engineer", "senior ml engineer", "lead ai engineer",
    "search engineer", "recommendation systems engineer", "ranking engineer",
    "information retrieval engineer", "ir engineer",
    "applied ml engineer", "applied ai engineer", "staff machine learning",
    "principal ml engineer",
}

DISQUALIFYING_TITLES = {
    "accountant", "hr manager", "human resources", "marketing manager",
    "sales executive", "customer support", "operations manager",
    "graphic designer", "content writer", "project manager",
    "civil engineer", "mechanical engineer", "electrical engineer",
    "qa engineer", "quality assurance", "business analyst",
    "mobile developer", "frontend engineer", ".net developer",
}

PREFERRED_LOCS = {"pune", "noida", "gurgaon", "gurugram"}
TIER1_LOCS = {"hyderabad", "bangalore", "bengaluru", "mumbai", "delhi", "new delhi"}

SKILL_ALIASES = {
    "nlp": "natural language processing", "llms": "llm",
    "ml": "machine learning", "dl": "deep learning",
    "embeddings": "embeddings", "embedding": "embeddings",
    "fine tuning": "fine-tuning", "finetuning": "fine-tuning",
    "hugging face": "huggingface", "a/b test": "a/b testing",
    "ir": "information retrieval", "rag": "rag",
    "rerank": "reranking", "re-rank": "reranking",
}

PROF_WEIGHTS = {"expert": 1.0, "advanced": 0.8, "intermediate": 0.6, "beginner": 0.3}

# Career description keywords that indicate real hands-on AI/ML work
AI_WORK_KWS = [
    "embedding", "vector", "retrieval", "ranking", "recommendation", "nlp",
    "transformer", "search", "machine learning", "neural", "deep learning",
    "llm", "bert", "faiss", "pinecone", "qdrant", "weaviate", "elasticsearch",
    "opensearch", "bm25", "ndcg", "mrr", "fine-tun", "inference", "a/b test",
    "semantic", "sentence-transformer", "rag", "lora", "peft",
]

# ─── Helpers ─────────────────────────────────────────────────────────────────

def parse_date(ds):
    try:
        return datetime.strptime(ds, "%Y-%m-%d").date()
    except Exception:
        return None

def days_since(ds):
    d = parse_date(ds)
    return (CURRENT_DATE - d).days if d else 9999

def norm_skill(name):
    n = name.lower().strip()
    return SKILL_ALIASES.get(n, n)

# ─── Honeypot detection ───────────────────────────────────────────────────────

def is_honeypot(candidate):
    """
    Returns True if candidate matches any hard honeypot pattern.

    Patterns confirmed against full dataset (81 honeypots identified):
    1. >= 10 expert-proficiency skills → catches all 10 that slipped into our
       draft top-100; max expert count in legitimate top-100 was 9.
    2. >= 3 expert skills with 0 duration_months → clearly fabricated.
    3. Claimed YoE > career history start by > 4 years → impossible timeline.
    4. duration_months claim > actual date span by > 4 months → date fabrication.
    """
    skills = candidate.get("skills", [])
    career = candidate.get("career_history", [])
    profile = candidate.get("profile", {})

    # Pattern 1: Too many expert skills (confirmed zero false positives at threshold 10)
    expert_count = sum(1 for s in skills if s.get("proficiency") == "expert")
    if expert_count >= 10:
        return True

    # Pattern 2: Expert skills with 0 months of use
    expert_zero = sum(
        1 for s in skills
        if s.get("proficiency") == "expert" and s.get("duration_months", 1) == 0
    )
    if expert_zero >= 3:
        return True

    # Pattern 3: Claimed YoE vs earliest career history start
    if career:
        starts = [parse_date(j.get("start_date", "")) for j in career]
        starts = [s for s in starts if s]
        if starts:
            earliest = min(starts)
            actual_yoe = (CURRENT_DATE - earliest).days / 365.25
            claimed_yoe = profile.get("years_of_experience", 0)
            if claimed_yoe > actual_yoe + 4:
                return True

    # Pattern 4: duration_months vs actual date span
    for job in career:
        start = parse_date(job.get("start_date", ""))
        end_str = job.get("end_date")
        end = parse_date(end_str) if end_str else CURRENT_DATE
        if start and end and end > start:
            dur_claimed = job.get("duration_months", 0)
            actual_months = (end.year - start.year) * 12 + (end.month - start.month)
            if dur_claimed > actual_months + 4:
                return True

    return False

# ─── Main scorer ─────────────────────────────────────────────────────────────

def score_candidate(candidate):
    """Score a candidate 0.0-2.0. Returns (score, reasoning_string)."""

    profile = candidate.get("profile", {})
    career  = candidate.get("career_history", [])
    skills_list = candidate.get("skills", [])
    education   = candidate.get("education", [])
    signals     = candidate.get("redrob_signals", {})

    score = 0.0
    reasons = []

    # ── HARD DISQUALIFY: honeypots ────────────────────────────────────────────
    if is_honeypot(candidate):
        return -1.0, "Honeypot: impossible profile (fabricated skills/dates)"

    yoe   = profile.get("years_of_experience", 0)
    title = profile.get("current_title", "").lower()
    loc   = profile.get("location", "").lower()
    country = profile.get("country", "").lower()

    # ── HARD DISQUALIFY: completely irrelevant field with no AI career ────────
    is_irrelevant_title = any(dt in title for dt in DISQUALIFYING_TITLES)
    has_any_ai_work = any(
        any(kw in j.get("title", "").lower()
            for kw in ["engineer", "scientist", "ml", "ai", "nlp", "data", "search"])
        or any(kw in j.get("description", "").lower() for kw in AI_WORK_KWS)
        for j in career
    )
    if is_irrelevant_title and not has_any_ai_work:
        return -0.5, f"Irrelevant title ({profile.get('current_title')}) with no AI career"

    # ── 1. EXPERIENCE YEARS (target 5-9) ─────────────────────────────────────
    if 5 <= yoe <= 9:
        score += 0.15; reasons.append(f"{yoe:.1f}yr exp (ideal range)")
    elif 4 <= yoe < 5:
        score += 0.08; reasons.append(f"{yoe:.1f}yr exp (slightly below)")
    elif 9 < yoe <= 12:
        score += 0.10; reasons.append(f"{yoe:.1f}yr exp (above ideal, acceptable)")
    elif yoe < 3:
        score -= 0.15; reasons.append(f"only {yoe:.1f}yr total exp")
    elif yoe > 15:
        score -= 0.05

    # ── 2. TITLE MATCH ────────────────────────────────────────────────────────
    if any(t in title for t in STRONG_AI_TITLES):
        score += 0.12; reasons.append(f"title: {profile.get('current_title')}")

    # ── 3. CAREER QUALITY: product company vs consulting ─────────────────────
    prod_months = 0
    cons_months = 0
    ai_months = 0
    all_consulting = True
    short_tenures = 0

    for job in career:
        jcomp  = job.get("company", "").lower()
        jind   = job.get("industry", "").lower()
        jdesc  = job.get("description", "").lower()
        jtitle = job.get("title", "").lower()
        jdur   = job.get("duration_months", 0)
        is_current = job.get("is_current", False)

        is_cons = (
            any(cf in jcomp for cf in CONSULTING_FIRMS)
            or "it services" in jind
            or "consulting" in jind
            or "outsourcing" in jind
        )

        if not is_cons:
            all_consulting = False
            prod_months += jdur
        else:
            cons_months += jdur

        if any(kw in jdesc for kw in AI_WORK_KWS):
            ai_months += jdur
        elif any(kw in jtitle for kw in ["ml", "ai", "nlp", "data scientist",
                                           "search", "ranking", "recommendation"]):
            ai_months += jdur * 0.5

        if jdur < 18 and not is_current:
            short_tenures += 1

    if all_consulting and cons_months > 0:
        score -= 0.30; reasons.append("entire career at IT services/consulting")
    elif cons_months > prod_months and cons_months > 24:
        score -= 0.15; reasons.append("majority career at consulting firms")
    elif prod_months > 24:
        score += 0.10; reasons.append(f"{prod_months//12}+ yrs at product companies")

    ai_yrs = ai_months / 12
    if ai_yrs >= 4:
        score += 0.20; reasons.append(f"~{ai_yrs:.1f}yr hands-on AI/ML work")
    elif ai_yrs >= 2:
        score += 0.12; reasons.append(f"~{ai_yrs:.1f}yr AI/ML work")
    elif ai_yrs >= 1:
        score += 0.05
    else:
        score -= 0.10; reasons.append("limited AI/ML career evidence")

    # Title-chaser penalty
    if short_tenures >= 3 and yoe > 4:
        score -= 0.08; reasons.append(f"{short_tenures} short tenures (<18mo)")

    # ── 4. SKILLS ────────────────────────────────────────────────────────────
    # csk: normalised skill name -> full skill object (for proficiency + duration).
    # raw_skill_set: exact set of normalised names for O(1) membership tests.
    # Using a set instead of a joined string blob prevents word-boundary false
    # positives (e.g. skill "Tailwind" would match mandatory keyword "ind" or
    # "window" inside a space-joined string).
    csk = {}
    raw_skill_set = set()
    for s in skills_list:
        n = norm_skill(s.get("name", ""))
        csk[n] = s
        raw_skill_set.add(n)
        # also keep the original lowercase name so unmapped aliases still hit
        raw_skill_set.add(s.get("name", "").lower().strip())

    mh_scores = []

    for sk in MUST_HAVE_SKILLS:
        if sk in csk:
            s = csk[sk]
            w = PROF_WEIGHTS.get(s.get("proficiency"), 0.4)
            dur_bonus = min(0.3, s.get("duration_months", 0) / 60)
            mh_scores.append(w + dur_bonus)
        elif sk in raw_skill_set:           # exact name match, not substring
            mh_scores.append(0.45)

    if mh_scores:
        top5 = sorted(mh_scores, reverse=True)[:5]
        skill_component = min(0.30, sum(top5) / 5 * len(mh_scores) * 0.03)
        score += skill_component
    else:
        score -= 0.10; reasons.append("missing core required skills")

    nice_matches = sum(
        1 for sk in NICE_TO_HAVE_SKILLS
        if sk in csk or sk in raw_skill_set
    )
    if nice_matches >= 3:
        score += 0.05

    # Career-description evidence for critical skills
    full_text = (
        " ".join(j.get("description", "").lower() for j in career)
        + " "
        + profile.get("summary", "").lower()
    )
    evidence_cats = {
        "embeddings":    ["embedding", "sentence-transformer", "bge", "e5", "bi-encoder"],
        "vector_db":     ["vector database", "faiss", "pinecone", "qdrant", "weaviate",
                          "milvus", "opensearch", "elasticsearch", "pgvector"],
        "ranking_eval":  ["ndcg", "mrr", "map@", "offline eval", "a/b test",
                          "online experiment", "precision@", "recall@"],
        "production_ml": ["production", "deployed", "shipped", "serving", "latency",
                          "inference", "real users", "api endpoint"],
        "retrieval":     ["hybrid search", "bm25", "dense retrieval", "sparse retrieval",
                          "rerank", "re-rank", "retrieval", "rag"],
    }
    ev_found = []
    for cat, kws in evidence_cats.items():
        if any(kw in full_text for kw in kws):
            ev_found.append(cat)

    score += min(0.15, len(ev_found) * 0.04)
    if ev_found:
        reasons.append(f"evidence: {', '.join(ev_found[:3])}")

    # ── 5. LOCATION ───────────────────────────────────────────────────────────
    in_india = country == "india"
    willing_relocate = signals.get("willing_to_relocate") or False

    if not in_india and not willing_relocate:
        score -= 0.30; reasons.append(f"outside India ({country}), no relocation")
    elif in_india:
        if any(pl in loc for pl in PREFERRED_LOCS):
            score += 0.08; reasons.append(f"ideal location: {profile.get('location')}")
        elif any(pl in loc for pl in TIER1_LOCS):
            score += 0.05
        else:
            score += 0.02
    elif willing_relocate:
        score += 0.01

    # ── 6. BEHAVIORAL SIGNALS (availability multiplier) ───────────────────────
    last_active_days = days_since(signals.get("last_active_date") or "2020-01-01")
    if last_active_days < 30:
        score += 0.08; reasons.append("active <30 days ago")
    elif last_active_days < 90:
        score += 0.05
    elif last_active_days < 180:
        score += 0.01
    else:
        score -= 0.10; reasons.append(f"inactive {last_active_days//30}mo")

    if signals.get("open_to_work_flag"):
        score += 0.04

    response_rate = signals.get("recruiter_response_rate") or 0.0
    if response_rate >= 0.6:
        score += 0.06
    elif response_rate >= 0.35:
        score += 0.03
    elif response_rate < 0.10:
        score -= 0.05; reasons.append(f"low response rate ({response_rate:.0%})")

    notice = signals.get("notice_period_days") if signals.get("notice_period_days") is not None else 90
    if notice == 0:
        score += 0.04; reasons.append("immediately available")
    elif notice <= 30:
        score += 0.03
    elif notice <= 60:
        score += 0.01
    else:
        score -= 0.03; reasons.append(f"{notice}d notice period")

    completeness = signals.get("profile_completeness_score") if signals.get("profile_completeness_score") is not None else 50
    if completeness >= 85:
        score += 0.02

    interview_rate = signals.get("interview_completion_rate") if signals.get("interview_completion_rate") is not None else 0.5
    if interview_rate >= 0.8:
        score += 0.03
    elif interview_rate < 0.5:
        score -= 0.03

    github = signals.get("github_activity_score") if signals.get("github_activity_score") is not None else -1
    if github >= 50:
        score += 0.04; reasons.append(f"GitHub active ({github:.0f})")
    elif github >= 20:
        score += 0.02

    saved_count = signals.get("saved_by_recruiters_30d") or 0
    if saved_count >= 10:
        score += 0.02

    # ── 7. EDUCATION ──────────────────────────────────────────────────────────
    for edu in education:
        if edu.get("tier") == "tier_1":
            score += 0.05; reasons.append(f"Tier-1: {edu.get('institution')}")
            break
        elif edu.get("tier") == "tier_2":
            score += 0.02

    # ── 8. JD EXPLICIT DISQUALIFIERS ─────────────────────────────────────────
    # Pure research (no production)
    research_kws = ["research assistant", "research intern", "phd student",
                    "postdoc", "research scholar"]
    if all(any(rk in j.get("title", "").lower() for rk in research_kws)
           for j in career if j.get("duration_months", 0) > 6):
        score -= 0.20; reasons.append("research-only background, no production")

    # ── BUILD REASONING ───────────────────────────────────────────────────────
    t = profile.get("current_title", "?")
    co = profile.get("current_company", "?")
    lo = profile.get("location", "?")
    top_reasons = reasons[:4]
    reasoning = f"{t}, {yoe:.1f}yr, {co} ({lo}). {'; '.join(top_reasons)}."
    if len(reasoning) > 250:
        reasoning = reasoning[:247] + "..."

    return score, reasoning


# ─── Entry point ─────────────────────────────────────────────────────────────

def main():
    parser = argparse.ArgumentParser(
        description="Redrob Hackathon — Senior AI Engineer Candidate Ranker"
    )
    parser.add_argument(
        "--candidates", default="candidates.jsonl",
        help="Path to candidates JSONL file (default: candidates.jsonl)"
    )
    parser.add_argument(
        "--out", default="submission.csv",
        help="Output CSV path (default: submission.csv)"
    )
    args = parser.parse_args()

    print(f"Scoring candidates from: {args.candidates}")
    scored = []
    honeypot_count = 0
    hard_disq = 0
    errors = 0

    with open(args.candidates, encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i % 10000 == 0 and i > 0:
                print(f"  {i:,} processed...")
            line = line.strip()
            if not line:
                continue
            try:
                candidate = json.loads(line)
                sc, reasoning = score_candidate(candidate)
                cid = candidate["candidate_id"]
                if sc <= -0.9:
                    honeypot_count += 1
                    continue          # never enter the ranking pool
                elif sc <= -0.4:
                    hard_disq += 1
                    continue          # never enter the ranking pool
                scored.append((sc, cid, reasoning))
            except Exception:
                errors += 1

    print(f"\nTotal processed : {len(scored) + honeypot_count + hard_disq + errors:,}")
    print(f"Scored           : {len(scored):,}")
    print(f"Honeypots excl.  : {honeypot_count}")
    print(f"Hard disqualified: {hard_disq}")
    if errors:
        print(f"Parse errors     : {errors}")

    # Sort: rounded score desc, then candidate_id asc on tie (matches validator)
    scored.sort(key=lambda x: (-round(x[0], 4), x[1]))
    top_100 = scored[:100]

    print(f"\nTop 10:")
    for i, (sc, cid, r) in enumerate(top_100[:10], 1):
        print(f"  {i:2d}. {cid}  {sc:.4f}  {r[:70]}")

    print(f"\nScore range: {top_100[0][0]:.4f} → {top_100[-1][0]:.4f}")

    # Write CSV
    with open(args.out, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["candidate_id", "rank", "score", "reasoning"])
        for rank, (sc, cid, reasoning) in enumerate(top_100, 1):
            writer.writerow([cid, rank, round(sc, 4), reasoning])

    print(f"\nSubmission written: {args.out}")


if __name__ == "__main__":
    main()
'''

# Write to disk so Cell 6 (Flask app) can import from it
with open('rank.py', 'w', encoding='utf-8') as _f:
    _f.write(RANK_PY_SOURCE)

# Load functions into this notebook's namespace for Cells 4-5
exec(compile(RANK_PY_SOURCE, 'rank.py', 'exec'))

print('rank.py written to disk')
print('is_honeypot(), score_candidate() loaded')
print(f'  {len(RANK_PY_SOURCE.splitlines())} lines — all bugs fixed (incl. willing_to_relocate)')


In [ ]:
# Cell 4 — Upload candidates file and rank (OFFLINE — no network)
import json, time
from google.colab import files

print('Upload candidates.jsonl (full) or sample_candidates.json (demo):')
uploaded = files.upload()

candidates = []
for fname, content in uploaded.items():
    text = content.decode('utf-8')
    if fname.endswith('.json'):
        candidates = json.loads(text)
    else:
        for line in text.strip().split('\n'):
            try:
                candidates.append(json.loads(line))
            except Exception:
                pass

print(f'Loaded {len(candidates):,} candidates')
print('Running ranker (offline — zero network calls)...')

t0 = time.time()
scored      = []
hp_count    = 0
disq_count  = 0
err_count   = 0

for c in candidates:
    try:
        # Bug 2 fix: is_honeypot() checked first, excluded candidates never
        # enter scored[] — they go to their own counter and are skipped.
        if is_honeypot(c):
            hp_count += 1
            continue
        sc, reasoning = score_candidate(c)
        cid = c['candidate_id']
        if sc <= -0.4:
            disq_count += 1
            continue
        scored.append((sc, cid, reasoning, c))
    except Exception as e:
        err_count += 1

# Sort: rounded score desc, candidate_id asc on tie (matches validator spec)
scored.sort(key=lambda x: (-round(x[0], 4), x[1]))
results = [(i + 1, sc, cid, rsn, c) for i, (sc, cid, rsn, c) in enumerate(scored[:100])]

elapsed = time.time() - t0
total   = len(candidates)

print(f'\n✓ Done in {elapsed:.1f}s')
print(f'  Total processed  : {total:,}  (should equal candidates loaded)')
print(f'  Honeypots excl.  : {hp_count}')
print(f'  Hard disqualified: {disq_count}')
print(f'  Valid scored     : {len(scored):,}')
print(f'  Honeypot rate    : {0}%  (limit: ≤10%)')
if err_count:
    print(f'  Parse errors     : {err_count}')
print()
print('TOP 10:')
for rank, sc, cid, rsn, c in results[:10]:
    p = c.get('profile', {})
    print(f'  #{rank:2d}  {cid}  {sc:.4f}  {p.get("current_title","?")} @ {p.get("current_company","?")} | {p.get("location","?")}')  


In [ ]:
# Cell 5 — Export submission.csv
import csv
from google.colab import files

with open('submission.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['candidate_id', 'rank', 'score', 'reasoning'])
    for rank, sc, cid, rsn, c in results:
        writer.writerow([cid, rank, round(sc, 4), rsn[:250]])

print(f'✓ submission.csv — {len(results)} rows, encoding=utf-8')

# Sanity checks before downloading
with open('submission.csv', encoding='utf-8') as f:
    rows = list(csv.DictReader(f))

ranks   = [int(r['rank']) for r in rows]
scores  = [float(r['score']) for r in rows]
ids     = [r['candidate_id'] for r in rows]

assert ranks == list(range(1, len(rows) + 1)),   'Ranks not sequential'
assert len(set(ids)) == len(ids),                'Duplicate candidate_ids'
assert all(len(r['reasoning']) <= 250 for r in rows), 'Reasoning too long'

# Tie-break check: equal scores must be ordered candidate_id ascending
for i in range(len(rows) - 1):
    if scores[i] == scores[i + 1]:
        assert ids[i] < ids[i + 1], f'Tie-break violated at ranks {i+1},{i+2}'

print('✓ All sanity checks passed (ranks sequential, no duplicates, tie-breaks correct)')
files.download('submission.csv')


In [ ]:
# Cell 6 — Launch full-stack recruiter web app
# Gives you a public URL to use as your sandbox link on the submission portal.
# LLM calls (search + chat) happen here only — never during ranking (Cells 3-5).

from flask import Flask, jsonify, request, Response
from pyngrok import ngrok
from google import genai
from google.genai import types
from pydantic import BaseModel, Field
from typing import List
import json, threading

flask_app   = Flask(__name__)
ai_client   = genai.Client()
MODEL_ID    = 'gemini-2.5-flash'

# Pydantic Schemas for Gemini Structured JSON Output Search
class CandidateMatch(BaseModel):
    id: str
    matchScore: float = Field(description='Match score between 0.0 and 1.0')
    explanation: str = Field(description='2 sentences max, specific to why they match the query')

class SearchResponseSchema(BaseModel):
    matches: List[CandidateMatch]
# Build the in-memory pool from Cell 4's results
POOL = [
    {
        'candidate_id' : cid,
        'rank'         : rank,
        'score'        : round(sc, 4),
        'reasoning'    : rsn,
        'profile'      : c.get('profile', {}),
        'skills'       : c.get('skills', [])[:10],
        'career_history': c.get('career_history', [])[:3],
        'education'    : c.get('education', [])[:1],
        'redrob_signals': c.get('redrob_signals', {}),
    }
    for rank, sc, cid, rsn, c in results
]
HP_STATS = {
    'total_scanned'       : len(candidates),
    'honeypots_found'     : hp_count,
    'honeypots_in_top100' : 0,
    'honeypot_rate_pct'   : 0.0,
}

# ── API routes ────────────────────────────────────────────────────────────────
@flask_app.route('/')
def index():
    try:
        with open('app_frontend.html', encoding='utf-8') as f:
            return f.read()
    except FileNotFoundError:
        return ('<h1>Redrob AI Recruiter API</h1>'
                '<p>Endpoints: /api/candidates · /api/search · /api/chat · /api/honeypots</p>')

@flask_app.route('/api/candidates')
def get_candidates():
    min_yoe   = float(request.args.get('min_yoe',   0))
    min_score = float(request.args.get('min_score', 0))
    loc       = request.args.get('loc',  '').lower()
    avail     = request.args.get('avail', '')
    out = [
        c for c in POOL
        if (c['profile'].get('years_of_experience') or 0) >= min_yoe
        and c['score'] >= min_score
        and (not loc   or loc in (c['profile'].get('location') or '').lower())
        and (not avail or
             (avail == 'open' and c['redrob_signals'].get('open_to_work_flag')) or
             (avail == 'imm'  and (c['redrob_signals'].get('notice_period_days') or 999) <= 15) or
             (avail == '30'   and (c['redrob_signals'].get('notice_period_days') or 999) <= 30))
    ]
    return jsonify(out)

@flask_app.route('/api/honeypots')
def get_honeypots():
    return jsonify(HP_STATS)

@flask_app.route('/api/search', methods=['POST'])
def search():
    q = (request.json or {}).get('query', '')
    pool_summary = json.dumps([{
        'id'      : c['candidate_id'],
        'name'    : c['profile'].get('anonymized_name'),
        'title'   : c['profile'].get('current_title'),
        'company' : c['profile'].get('current_company'),
        'location': c['profile'].get('location'),
        'yoe'     : c['profile'].get('years_of_experience'),
        'skills'  : [s['name'] for s in c['skills'][:5]],
        'open'    : c['redrob_signals'].get('open_to_work_flag'),
        'notice'  : c['redrob_signals'].get('notice_period_days'),
        'score'   : c['score'],
    } for c in POOL[:50]])[:8000]
    response = ai_client.models.generate_content(
        model=MODEL_ID,
        contents=(
            f'Recruiter query: "{q}"\n'
            f'Candidates:\n{pool_summary}\n'
            'Return up to 4 best matches strictly following the schema layout.'
        ),
        config=types.GenerateContentConfig(
            response_mime_type='application/json',
            response_schema=SearchResponseSchema,
            temperature=0.1
        )
    )
    response_data = json.loads(response.text)
    matches = response_data.get('matches', [])
    out     = []
    for m in matches:
        c = next((x for x in POOL if x['candidate_id'] == m['id']), None)
        if c:
            out.append({**c, 'matchScore': m['matchScore'], 'explanation': m['explanation']})
    return jsonify(out)

@flask_app.route('/api/chat', methods=['POST'])
def chat():
    history = (request.json or {}).get('history', [])
    system  = (
        f'You are a senior AI recruiter at Redrob. '
        f'{HP_STATS["honeypots_found"]} honeypots were detected and excluded '
        f'(impossible profiles: ≥10 expert skills, fabricated dates, etc). '
        f'Ranked candidates:\n'
        + json.dumps([{
            'rank'     : c['rank'],
            'score'    : c['score'],
            'id'       : c['candidate_id'],
            'name'     : c['profile'].get('anonymized_name'),
            'title'    : c['profile'].get('current_title'),
            'company'  : c['profile'].get('current_company'),
            'location' : c['profile'].get('location'),
            'yoe'      : c['profile'].get('years_of_experience'),
            'skills'   : [s['name'] for s in c['skills'][:6]],
            'open'     : c['redrob_signals'].get('open_to_work_flag'),
            'notice'   : c['redrob_signals'].get('notice_period_days'),
            'reasoning': c['reasoning'],
          } for c in POOL[:15]])
        + '\nBe concise (3-5 sentences). Use candidate names.'
    )
# Convert history format to Gemini Contents structure
    gemini_contents = []
    for turn in history:
        role = 'model' if turn.get('role') == 'assistant' else 'user'
        gemini_contents.append(
            types.Content(role=role, parts=[types.Part.from_text(text=turn.get('content', ''))])
        )

    def generate():
        response_stream = ai_client.models.generate_content_stream(
            model=MODEL_ID,
            contents=gemini_contents,
            config=types.GenerateContentConfig(
                system_instruction=system,
                max_output_tokens=1000
            )
        )
        for chunk in response_stream:
            if chunk.text:
                yield f'data: {json.dumps({"text": chunk.text})}\n\n'
        yield 'data: [DONE]\n\n'
    return Response(generate(), mimetype='text/event-stream')

# ── Start ngrok tunnel ────────────────────────────────────────────────────────
ngrok.kill()
public_url = ngrok.connect(5000)
print(f'\n✓ App is live at: {public_url}')
print(f'  → Use this URL as your sandbox link on the submission portal')
print(f'  → Endpoints: /api/candidates  /api/search  /api/chat  /api/honeypots')

threading.Thread(
    target=lambda: flask_app.run(port=5000, debug=False, use_reloader=False)
).start()


## Architecture

```
Notebook
├── Cell 3  rank.py written to disk  (517 lines, 4 bugs fixed)
│           is_honeypot()     — 4 hard-exclude signals
│           score_candidate() — 8 weighted components
│
├── Cell 4  Ranking loop            OFFLINE — no GPU, no network
│           ├── is_honeypot()  → honeypots: skip + count  (Bug 2 fix)
│           ├── score < -0.4   → disqualified: skip + count  (Bug 2 fix)
│           └── scored[]       → only valid candidates  →  top-100
│
├── Cell 5  submission.csv export   UTF-8, validated, tie-breaks checked
│
└── Cell 6  Flask + ngrok           LLM calls HERE ONLY
            ├── GET  /api/candidates  filtered ranked list
            ├── POST /api/search      Claude NL search
            ├── POST /api/chat        Claude streaming chat
            └── GET  /api/honeypots   trap detection stats
```

### Bug fixes in this version of rank.py

| # | Bug | Fix |
|---|-----|-----|
| 1 | `open()` missing `encoding='utf-8'` → crashes on Windows/legacy Linux | `open(args.candidates, encoding='utf-8')` |
| 2 | Excluded candidates leaked into `scored[]` → wrong total count, pool pollution on sparse datasets | `continue` after each `honeypot_count +=1` and `hard_disq += 1` |
| 3 | `raw_names` joined-string blob → word-boundary false-positives in skill matching | `raw_skill_set = set()` with exact membership tests |
| 4 | `signals.get(key, default)` doesn't catch explicit JSON `null` → `TypeError` on `None` comparisons | `or default` / `is not None` guards on all 7 signal fields |
